In [1]:
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
# torch.multiprocessing.set_start_method('spawn', force=True)
import time
import importlib
import torch.optim as optim
from torchsummary import summary
import os
import statistics

os.chdir("../")
from src.data_preparation import SimulatedDataset, collate_function, load_real_data
from src.transformers import MiniTransformer
import src.transformers as transformerFunctions
from src.transformers import init_weights_recursive
from src.transformers import print_parameters
from src.transformers import create_custom_mask_pair,create_custom_mask_pred, create_distance_to_end_matrix, create_pairwise_distance_matrix
from src.evaluation import calculate_bench1_loss, calculate_bench2_loss, calculate_repeat_loss, calculate_regression_loss, evaluate_mini_transformer
from src.statistical_testing import statistical_testing, print_p_values, plot_context_predindex_pair_effect, get_context_predindex_pair_effect
import torch.autograd.profiler as profiler

device = torch.device("cpu")

In [2]:

if __name__ == '__main__':
    
    bench_repeat_loss_list = []
    regression_loss_total_list = []
    model_loss_total_list = []
    regression_loss_predindex_list = []
    model_loss_predindex_list = []
    bench1_loss_list = []
    bench2_loss_list = []
    bench2loss_predindex_list = []
    bench_repeat_loss_predindex_list = []
    benchloss_predindex_list = []
    models = []
    


    # Hyperparameters
    # data_str = "ghq_b_sum"
    # data_str = "ghq_sum"
    data_str = "simulation"
    batch_size = 1           # Batch size for loading data
    dk = 1                  # d_k
    dv = 1                  # d_v
    nheads = 16             # number of heads
    ncum = 2                 # number of cumulants
    maxlen = 10             # maximum length of the sequence
    learning_rate = 5e-4
    lambda_l2 = 1e-3
    EPOCHS = 100
    target_sample_size = 7
    nrepp = 10
    seeds = [0, 1, 11, 42, 123, 999, 1337, 2025, 9999, 12345]
    
    
    for iteration in range(len(seeds)):
        
        # Set the random seed for reproducibility
        seed = seeds[iteration]
        torch.manual_seed(seed)

        if data_str == "simulation":
            n = 200
            p = 10
            maxlen = 10
            # Create Dataset and DataLoader
            train_dataset = SimulatedDataset(n, p, maxlen=maxlen, device = device).data
            eval_dataset = SimulatedDataset(1000, p, maxlen=4, device = device).data
            predindex = 2
            
        else:
            # load real data
            data, maxlen = load_real_data(data_str)
            
            n = int(0.8*len(data))
            p = data[0].shape[1]
            print("Sample size: ", len(data))
            
            train_dataset = data[:n]
            eval_dataset = data[n:]
            predindex = 9

        

        mask_pairwise = create_custom_mask_pair(maxlen, device)
        mask_pred = create_custom_mask_pred(maxlen, device)
        distance_to_end_matrix = create_distance_to_end_matrix(maxlen, device)
        pairwise_distance_matrix = create_pairwise_distance_matrix(maxlen, device)

    
        dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_function,  num_workers=0)
        
        models.append(MiniTransformer(p, nheads, dk, dv, ncum, mask_pairwise,mask_pred, pairwise_distance_matrix, distance_to_end_matrix,  device))

        models[iteration].apply(init_weights_recursive)
        models[iteration].to(device)

        # model = torch.compile(model)
        # Start the timer
        start_time = time.time()

        # Define optimizer
        # optimizer = optim.Adam(models[iteration].parameters(), lr= learning_rate, weight_decay=lambda_l2)
        optimizer = optim.Adam(models[iteration].parameters(), lr= learning_rate)
        
        print("Number of Parameters", transformerFunctions.count_parameters(models[iteration]))
        
        run_path = transformerFunctions.train_mini_transformer(models[iteration], dataloader, optimizer, lambda_l2, EPOCHS, device)


        # End the timer
        end_time = time.time()

        # Calculate and print the execution time
        execution_time = end_time - start_time 
        print(f"Execution time: {execution_time:.6f} seconds")
        
        # Evaluate the model
        
        eval_dataloader = DataLoader(eval_dataset, batch_size=1, shuffle=True, collate_fn=collate_function, num_workers=0)
       
    
    
        dimave, bench1_loss, benchloss_predindex = calculate_bench1_loss(train_dataset, eval_dataset, predindex)
        regression_loss_predindex, regression_loss_total = calculate_regression_loss(train_dataset, eval_dataset, predindex)
        model_loss_predindex, model_loss_total = evaluate_mini_transformer(eval_dataloader, models[iteration], predindex)
        
        
        print("baseline average loss: ", bench1_loss)

        
        # Evaluate the model
        if data_str == "simulation":
            bench2loss, bench2loss_predindex = calculate_bench2_loss(train_dataset, eval_dataset, dimave)
            print("baseline informed loss: ", bench2loss)
            bench2_loss_list.append(bench2loss)
            bench2loss_predindex_list.append(bench2loss_predindex)
            

        bench_repeat, bench_repeat_predindex = calculate_repeat_loss(eval_dataset, predindex)    
        print("baseline repeat: ", bench_repeat)
        
          
        print("regression loss total: ", regression_loss_total)
        print("model loss total: ", model_loss_total.item(), "\n") 
        
        
        if data_str == "simulation":
            print("baseline informed loss predindex: ", bench2loss_predindex)
        
            
        print("baseline average loss predindex: ", benchloss_predindex)
        print("baseline repeat predindex: ", bench_repeat_predindex)    
        print("regression loss predindex: ", regression_loss_predindex)
        print("model loss predindex: ", model_loss_predindex.item()) 

        
        bench_repeat_loss_list.append(bench_repeat)
        regression_loss_total_list.append(regression_loss_total)
        model_loss_total_list.append(model_loss_total.item())
        bench2loss_predindex_list.append(bench2loss_predindex)
        benchloss_predindex_list.append(benchloss_predindex)
        bench_repeat_loss_predindex_list.append(bench_repeat_predindex)
        regression_loss_predindex_list.append(regression_loss_predindex)
        model_loss_predindex_list.append(model_loss_predindex.item())
        bench1_loss_list.append(bench1_loss)
        
        


with open(f"./notebooks/{data_str}_results_n={n}_batch_size={batch_size}_p={p}_ncum={ncum}_nheads={nheads}_epochs={EPOCHS}.txt", "a") as f:
    # First line
    f.write(f"{data_str} results n = {n}\n")

    # Single-line writes with three-decimal formatting:
    f.write(f"baseline repeat: {statistics.mean(bench_repeat_loss_list):.3f} ± {statistics.stdev(bench_repeat_loss_list):.3f}\n")
    f.write(f"baseline average loss: {statistics.mean(bench1_loss_list):.3f} ± {statistics.stdev(bench1_loss_list):.3f}\n")
    f.write(f"baseline informed loss: {statistics.mean(bench2_loss_list):.3f} ± {statistics.stdev(bench2_loss_list):.3f}\n")
    f.write(f"regression loss total: {statistics.mean(regression_loss_total_list):.3f} ± {statistics.stdev(regression_loss_total_list):.3f}\n")
    f.write(f"model loss total: {statistics.mean(model_loss_total_list):.3f} ± {statistics.stdev(model_loss_total_list):.3f}\n\n")

    
    f.write(f"baseline repeat predindex: {statistics.mean(bench_repeat_loss_predindex_list):.3f} ± {statistics.stdev(bench_repeat_loss_list):.3f}\n")
    f.write(f"baseline average loss predindex: {statistics.mean(benchloss_predindex_list):.3f} ± {statistics.stdev(benchloss_predindex_list):.3f}\n")
 
    f.write(f"regression loss predindex: {statistics.mean(regression_loss_predindex_list):.3f} ± {statistics.stdev(regression_loss_predindex_list):.3f}\n")
    if data_str == "simulation":
        f.write(f"baseline informed loss predindex: {statistics.mean(bench2loss_predindex_list):.3f} ± {statistics.stdev(bench2loss_predindex_list):.3f}\n")
    f.write(f"model loss predindex: {statistics.mean(model_loss_predindex_list):.3f} ± {statistics.stdev(model_loss_predindex_list):.3f}\n")

    # Extra newlines at the end
    f.write("\n\n")

Number of Parameters 592
EPOCH 1:
avg_loss: 19.70933
EPOCH 11:
avg_loss: 9.12875
EPOCH 21:
avg_loss: 9.08459
EPOCH 31:
avg_loss: 9.05630
EPOCH 41:
avg_loss: 9.02259
EPOCH 51:
avg_loss: 8.98767
EPOCH 61:
avg_loss: 8.95977
EPOCH 71:
avg_loss: 8.93172
EPOCH 81:
avg_loss: 8.90263
EPOCH 91:
avg_loss: 8.83769
Execution time: 15.079407 seconds
baseline average loss:  0.2149399707466364
baseline informed loss:  0.20379027724266052
baseline repeat:  0.4531000071018934
regression loss total:  0.20683662575955405
model loss total:  0.2075244039297104 

baseline informed loss predindex:  0.13199999928474426
baseline average loss predindex:  0.24350597009435296
baseline repeat predindex:  0.68
regression loss predindex:  0.13263466889421868
model loss predindex:  0.13854235410690308
Number of Parameters 592
EPOCH 1:
avg_loss: 19.01180
EPOCH 11:
avg_loss: 8.50461
EPOCH 21:
avg_loss: 8.45619
EPOCH 31:
avg_loss: 8.41150
EPOCH 41:
avg_loss: 8.36814
EPOCH 51:
avg_loss: 8.31382
EPOCH 61:
avg_loss: 8.2436